# Build your EPANET AI Agent

This notebook is a small demonstration of how to use EPANET MCP server in a large language model agent.

First, we are going to set up the api keys and base url of the provider and the MCP server:

In [13]:

import os
import sys
from dotenv import load_dotenv
load_dotenv()

from langchain_mcp_adapters.client import MultiServerMCPClient

api_key = os.getenv('OPENAI_API_KEY', None)
if api_key is None:
    print("WARNING: No API key provided")
base_url = os.getenv('OPENAI_BASE_URL', None)
if base_url is None:
    print("WARNING: No base URL provided")

client = MultiServerMCPClient(
    {
        "epanet": {
            "transport": "stdio",
            "command": "uvx",
            "args": ["epanet-mcp-server@v1.0.1"],
        }
    }
)
tools = await client.get_tools()
print("Available tools:", [t.name for t in tools])

Available tools: ['load_network', 'unload_network', 'list_networks', 'list_bundled_networks', 'get_network_summary', 'get_nodes', 'get_links', 'get_patterns', 'get_controls', 'get_curves', 'get_options', 'run_hydraulic_simulation', 'run_quality_simulation', 'run_full_simulation', 'get_pressure_at_time', 'get_flow_at_time', 'save_network', 'set_node_base_demand', 'set_pattern', 'add_pattern', 'set_pipe_diameter', 'set_pipe_roughness', 'set_pipe_status', 'set_pipe_length', 'set_pump_status', 'set_pump_speed', 'set_pump_head_curve', 'set_valve_setting', 'set_valve_status', 'set_tank_parameters', 'set_reservoir_head', 'set_simulation_duration', 'set_hydraulic_timestep', 'set_pattern_timestep', 'set_quality_timestep', 'set_quality_type', 'add_control', 'delete_control', 'create_demand_perturbation', 'create_leakage_event', 'create_contamination_event', 'create_pressure_change_scenario', 'create_pump_control_scenario', 'create_valve_control_scenario', 'create_multi_failure_scenario']


In [5]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5.6-luna",
    api_key=api_key,
    base_url=base_url,
)

# Start a conversation

Then, we will use the MCP server to start a session.

With a stateful EPANET MCP server using stdio, all tool calls must share the same MCP session. Otherwise, load_network loads the network into one server process, but subsequent calls may connect to a different process with an empty network registry.

The MCP session remains open only during `run_epanet_agent()`. If you want to execute multiple prompts against the same loaded network, keep the `async with client.session("epanet")` block open across those agent invocations.

## List all available networks

In [6]:
from langchain_mcp_adapters.tools import load_mcp_tools


prompt = "List all the available networks."

async with client.session("epanet") as session:
    # All tools share the same MCP session
    tools = await load_mcp_tools(session)

    agent = create_agent(
        model=model,
        tools=tools,
    )

    result = await agent.ainvoke({
        "messages": [
            {"role": "user", "content": prompt}
        ]
    })


In [7]:
for message in result["messages"]:
    # show type of message
    message_type = type(message).__name__
    print("\t", message_type)
    if message_type == "HumanMessage":
        print(message.content)
    elif message_type == "AIMessage":
        if len(message.tool_calls) != 0:
            print("tool calls:", message.tool_calls)
        if message.content != "":
            print(message.content)
    print("")

	 HumanMessage
List all the available networks.

	 AIMessage
tool calls: [{'name': 'list_bundled_networks', 'args': {}, 'id': 'call_YrLOBIxOeI2bgTfYZebyIqCL', 'type': 'tool_call'}]

	 ToolMessage

	 AIMessage
Available bundled networks:

- `L-TOWN.inp`
- `asce-tf-wdst/Anytown.inp`
- `asce-tf-wdst/BWSN_Network_1.inp`
- `asce-tf-wdst/BWSN_Network_2.inp`
- `asce-tf-wdst/Balerma.inp`
- `asce-tf-wdst/Battle of the Calibration Networks System.inp`
- `asce-tf-wdst/Extended Hanoi.inp`
- `asce-tf-wdst/Hanoi.inp`
- `asce-tf-wdst/Jilin including water quality.inp`
- `asce-tf-wdst/KL.inp`
- `asce-tf-wdst/MICROPOLIS_v1.inp`
- `asce-tf-wdst/Modified New York Tunnels including water quality.inp`
- `asce-tf-wdst/Net1.inp`
- `asce-tf-wdst/Net1broken.inp`
- `asce-tf-wdst/Net2.inp`
- `asce-tf-wdst/Net3.inp`
- `asce-tf-wdst/Net3_trace.inp`
- `asce-tf-wdst/New York Tunnels including water quality.inp`
- `asce-tf-wdst/RuralNetwork.inp`
- `asce-tf-wdst/ZJ.inp`
- `asce-tf-wdst/exnet-3.inp`
- `asce-tf-wdst/fos

## Run hydraulic simulation

The proposed example attempts to demonstrate how to tackle hydraulic simulations and hydraulic scenarios by asking to the language model a comparison between different simulations.

In [8]:
from langchain_mcp_adapters.tools import load_mcp_tools


prompt = "Load Net1.inp as a network named Net1. Report its default hydraulic simulation duration. Run a hydraulic simulation at that default duration and identify the junction with the lowest pressure across all simulation time steps, including its pressure and the time it occurs. Then set the simulation duration to 6 hours, rerun the hydraulic simulation, and report the lowest-pressure junction, pressure, and time for that run. Keep the network session at the 6-hour duration. Do not count reservoirs or tanks when finding the lowest-pressure junction."

async with client.session("epanet") as session:
    # All tools share the same MCP session
    tools = await load_mcp_tools(session)

    agent = create_agent(
        model=model,
        tools=tools,
    )

    result = await agent.ainvoke({
        "messages": [
            {"role": "user", "content": prompt}
        ]
    })


In [9]:
for message in result["messages"]:
    # show type of message
    message_type = type(message).__name__
    print("\t", message_type)
    if message_type == "HumanMessage":
        print(message.content)
    elif message_type == "AIMessage":
        if len(message.tool_calls) != 0:
            print("tool calls:", message.tool_calls)
        if message.content != "":
            print(message.content)
    print("")

	 HumanMessage
Load Net1.inp as a network named Net1. Report its default hydraulic simulation duration. Run a hydraulic simulation at that default duration and identify the junction with the lowest pressure across all simulation time steps, including its pressure and the time it occurs. Then set the simulation duration to 6 hours, rerun the hydraulic simulation, and report the lowest-pressure junction, pressure, and time for that run. Keep the network session at the 6-hour duration. Do not count reservoirs or tanks when finding the lowest-pressure junction.

	 AIMessage
tool calls: [{'name': 'load_network', 'args': {'path': 'Net1.inp', 'network_id': 'Net1'}, 'id': 'call_d72279582e344821829940b5627bd83b', 'type': 'tool_call'}]

	 ToolMessage

	 AIMessage
tool calls: [{'name': 'run_hydraulic_simulation', 'args': {'network_id': 'Net1'}, 'id': 'call_rRM5sNMN8oPmvCI7iPFDaVo2', 'type': 'tool_call'}]

	 ToolMessage

	 AIMessage
tool calls: [{'name': 'set_simulation_duration', 'args': {'networ